# Shapedb Search Session

This notebook prepares a PDB-derived ligand shape file and launches a shapedb search against the Enamine database.

- It saves the ligand segment `LIG` from the PDB as `.mol2`.
- Then it converts and centers that `.mol2` to `_centered.sdf` using `func/shapedb/convert_and_center_mol2.py`.
- Finally it uses the generated `.sdf` for the shapedb search.

In [2]:
## input a pdb file with desing truncated structure
import os
from pathlib import Path
import subprocess
import sys
import pymol
from pymol import cmd

def run_cmd(cmd):
	result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
	if result.stdout:
		print(result.stdout, end="")
	if result.stderr:
		print(result.stderr, file=sys.stderr, end="")
	return result.returncode

#########
working_dir ="/pi/summer.thyme-umw/Ji_rosetta_discovery/"
enamine_database  = "/pi/summer.thyme-umw/enamine-REAL-2.6billion"
motifs_file=Path(enamine_database) / "FINAL_motifs_list_filtered_2_3_2023.motifs"
########





In [ ]:

pdb_path = Path(working_dir) / "input_pdb"
shapedb_output_root = Path(working_dir) / "out/shapedb"
convert_script = Path(working_dir) / "func" / "shapedb" / "convert_and_center_mol2.py"

print("working_dir:", working_dir)
print("pdb_path:", pdb_path)
print("shapedb_output_root:", shapedb_output_root)

if not pdb_path.exists():
    raise FileNotFoundError(f"PDB file not found: {pdb_path}")

for pdb_file in pdb_path.glob("*.pdb"):
    stem = pdb_file.stem
    shapedb_output = shapedb_output_root / stem
    ligand_mol2 = shapedb_output / f"{stem}.mol2"
    ligand_sdf = shapedb_output / f"{stem}_centered.sdf"

    print("Processing PDB:", pdb_file)
    shapedb_output.mkdir(parents=True, exist_ok=True)

    cmd.load(str(pdb_file), object='agonist')
    cmd.remove('chain R')
    cmd.save(str(ligand_mol2), 'segid LIG', format='mol2')
    print("Saved MOL2:", ligand_mol2)

    if not ligand_sdf.exists():
        print("Converting MOL2 to centered SDF:", ligand_mol2)
        subprocess.run(["python", str(convert_script), str(shapedb_output)], check=True)
    else:
        print("Centered SDF already exists:", ligand_sdf)

    print("Output SDF:", ligand_sdf)

working_dir: /pi/summer.thyme-umw/Ji_rosetta_discovery/
pdb_path: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb
shapedb_output_root: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb
Processing PDB: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/orexin_lemborexan.pdb
Saved MOL2: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexan/orexin_lemborexan.mol2
Converting MOL2 to centered SDF: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexan/orexin_lemborexan.mol2
Processed: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexan/orexin_lemborexan.mol2 -> /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexan/orexin_lemborexan_centered.sdf
Done!
Output SDF: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexan/orexin_lemborexan_centered.sdf
Processing PDB: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/orexin_state3.pdb
Saved MOL2: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shape

In [ ]:
# check your output sdf, if it's good set this to True to execute the shapedb search.
run_search = False

ligand_mol2 =  "/pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3_centered.sdf"
shapedb_output = "/pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/"
if run_search:
    shapedb_controller = Path(working_dir) / "func" / "shapedb" / "nnsearch_controller.py"
    shapedb_cmd = ["bsub -q long -n 8 -W 8:00 -R \"rusage[mem=5000]\" -J shapedb_search_ctrl" ,
        "python",
        str(shapedb_controller),
        str(ligand_mol2),
        "--working-dir",
        str(shapedb_output),
        "--realm-dir","/pi/summer.thyme-umw/Ji_rosetta_discovery",
        "-n", "3000000", "-j", "500", "--min-chunk", "0", "--max-chunk", "53084",

    ]
    print("Executing shapedb search:", " ".join(shapedb_cmd))
    run_cmd(" ".join(shapedb_cmd))
else:
    print("Search not executed.")
    print("Set run_search = True and rerun this cell when you want to submit shapedb.")


Executing shapedb search: bsub -q long -n 8 -W 8:00 -R "rusage[mem=5000]" -J shapedb_search_ctrl python /pi/summer.thyme-umw/Ji_rosetta_discovery/func/shapedb/nnsearch_controller.py /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3_centered.sdf --working-dir /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/ --realm-dir /pi/summer.thyme-umw/Ji_rosetta_discovery -n 3000000 -j 500 --min-chunk 0 --max-chunk 53084
Job <940007> is submitted to queue <long>.


WARN: No span specified with (-n 8). Defaulting to span[hosts=1] (single host). See https://tinyurl.com/3rctk5j9
INFO: Total memory requested is 40000 MB (8 cores x 5000 MB)


In [ ]:
run_cleanup = False
top_ligands_list = Path(shapedb_output) / "combined_results/combined_best_3000000_chunks_00000_53084.txt"
if run_cleanup:
    cleanup_cmd = ["bsub -q long -n 32 -W 168:00 -R \"rusage[mem=2000]\" -J param_ligand_ctrl" ,
        "python", 
        "./func/discovery_test_params_preparation/prepare_test_params_directories_controller.py",
        str(shapedb_output),
        str(top_ligands_list),
            "--max-workers", "30",
    ]
    
    run_cmd(" ".join(cleanup_cmd))

Job <823820> is submitted to queue <long>.


WARN: No span specified with (-n 32). Defaulting to span[hosts=1] (single host). See https://tinyurl.com/3rctk5j9
INFO: Total memory requested is 64000 MB (32 cores x 2000 MB)


In [ ]:
target_pdb="/pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/orexin_state4.pdb"
motifs_file = "/pi/summer.thyme-umw/Ji_rosetta_discovery/motifs/FINAL_motifs_list_filtered_2_3_2023.motifs"
Rosetta_scripts_dir = Path(working_dir) / "func" / "rosetta"
shapedb_output_dir = Path(working_dir) / "out/shapedb/orexin_state4/" 
rosetta_run =False 
if rosetta_run:
    #comma-separated (no spaces) of all residue indices (in Rosetta format) intended to be considered for discovery
    anchor_residue_string="134"

    atr_cutoff= "-2"
    rep_cutoff= "150"
    ddg_cutoff= "-9"

    extra_args_file = ""

    Rosetta_scripts_dir = Path(working_dir) / "func" / "rosetta"
    rosetta_cmd  = ["bsub -q long -n 1 -W 168:00 -R \"rusage[mem=2000]\" -J rosetta_ctrl" , 
                    "python",
                    str(Rosetta_scripts_dir) + "/run_ligand_discovery_search_controller.py",
                    target_pdb,# need absolut path for this
                    anchor_residue_string,
                    motifs_file, # need absolut path for this
                    "/pi/summer.thyme-umw/Ji_rosetta_discovery",
                    str(shapedb_output_dir), 
                    atr_cutoff,
                    rep_cutoff,
                    ddg_cutoff,
                    'false', # clobber is set to False by default, change to True if you want to clobber existing discovery runs
                    extra_args_file]
    run_cmd(" ".join(rosetta_cmd))


Job <808056> is submitted to queue <long>.
